In [ ]:
import os, sys
project_root = 'C:\min\icesat-2'
sys.path.append(project_root)

import icepyx as ipx
import numpy as np
import xarray as xr
import pandas as pd
import h5py
import os, json
from pprint import pprint
import dask.dataframe as dd
import rioxarray as rxr
import rasterio
import rasterio
from pyproj import Transformer

In [ ]:
import glob
import re
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import MultipleLocator


from readers.add_atl08_info import add_atl08_classed_flag
from readers.get_ATL03_x_atc import get_atl03_x_atc
from readers.read_HDF5_ATL03 import read_hdf5_atl03_beam_h5py

def read_hdf5_atl03_beam_h5py(filename, beam, verbose=False):
    """
    ATL03 原始数据读取
    Args:
        filename (str): h5文件路径
        beam (str): 光束
        verbose (bool): 输出HDF5信息

    Returns:
        返回ATL03光子数据的heights和geolocation信息
    """

    # 打开HDF5文件进行读取
    file_id = h5py.File(os.path.expanduser(filename), 'r')

    # 输出HDF5文件信息
    if verbose:
        print(file_id.filename)
        print(list(file_id.keys()))
        print(list(file_id['METADATA'].keys()))

    # 为ICESat-2 ATL03变量和属性分配python字典
    atl03_mds = {}

    # 读取文件中每个输入光束
    beams = [k for k in file_id.keys() if bool(re.match('gt\\d[lr]', k))]
    if beam not in beams:
        print('请填入正确的光束代码')
        return

    atl03_mds['heights'] = {}
    atl03_mds['geolocation'] = {}
    atl03_mds['bckgrd_atlas'] = {}

    # -- 获取每个HDF5变量
    # -- ICESat-2 Measurement Group
    for key, val in file_id[beam]['heights'].items():
        atl03_mds['heights'][key] = val[:]

    # -- ICESat-2 Geolocation Group
    for key, val in file_id[beam]['geolocation'].items():
        atl03_mds['geolocation'][key] = val[:]

    for key, val in file_id[beam]['bckgrd_atlas'].items():
        atl03_mds['bckgrd_atlas'][key] = val[:]

    return atl03_mds

def get_atl03_x_atc(atl03_mds):
    val = atl03_mds

    # 初始化
    val['heights']['x_atc'] = np.zeros_like(val['heights']['h_ph']) + np.nan
    val['heights']['y_atc'] = np.zeros_like(val['heights']['h_ph']) + np.nan
    val['geolocation']['ref_elev_all'] = np.zeros_like(val['heights']['h_ph'])

    # -- ATL03 Segment ID
    segment_id = val['geolocation']['segment_id']
    # -- 分段中的第一个光子（转换为基于0的索引）
    segment_index_begin = val['geolocation']['ph_index_beg'] - 1
    # -- 分段中的光子事件数
    segment_pe_count = val['geolocation']['segment_ph_cnt']
    # -- 每个ATL03段的沿轨道距离
    segment_distance = val['geolocation']['segment_dist_x']
    # -- 每个ATL03段的轨道长度
    segment_length = val['geolocation']['segment_length']

    # -- 对ATL03段进行迭代，以计算40m的平均值
    # -- 在ATL03中基于1的索引：无效==0
    # -- 此处为基于0的索引：无效==-1
    segment_indices, = np.nonzero((segment_index_begin[:-1] >= 0) &
                                  (segment_index_begin[1:] >= 0))
    for j in segment_indices:
        # -- j 段索引
        idx = segment_index_begin[j]
        # -- 分段中的光子数（使用2个ATL03分段）
        c1 = np.copy(segment_pe_count[j])
        c2 = np.copy(segment_pe_count[j + 1])
        cnt = c1 + c2

        # -- 沿轨道和跨轨道距离
        # -- 获取当前段光子列表，idx当前段(j)第一个光子数量，c1当前段光子数量，idx+c1当前段长度
        distance_along_x = np.copy(val['heights']['dist_ph_along'][idx: idx + cnt])
        ref_elev = np.copy(val['geolocation']['ref_elev'][j])
        # -- 给当前段的光子加上当前段沿轨道距离
        distance_along_x[:c1] += segment_distance[j]
        distance_along_x[c1:] += segment_distance[j + 1]
        distance_along_y = np.copy(val['heights']['dist_ph_across'][idx: idx + cnt])

        val['heights']['x_atc'][idx: idx + cnt] = distance_along_x
        val['heights']['y_atc'][idx: idx + cnt] = distance_along_y
        val['geolocation']['ref_elev_all'][idx: idx + c1] += ref_elev

def read_data(filepath, beam, mask_lat, mask_lon):
    """
    读取数据，返回沿轨道距离和高程距离
    :param filepath: h5文件路径
    :param beam: 轨道光束
    :param mask_lat: 维度范围
    :param mask_lon: 经度范围
    :return:
    """
    atl03_file = glob(filepath)
    is2_atl03_mds = read_hdf5_atl03_beam(atl03_file[0], beam=beam, verbose=False)
    # 添加沿轨道距离到数据中
    get_atl03_x_atc(is2_atl03_mds)
    # 选择范围
    d3 = is2_atl03_mds
    subset1 = (d3['heights']['lat_ph'] >= min(mask_lat)) & (d3['heights']['lat_ph'] <= max(mask_lat))
    print(subset1.shape, 'sss')
    zz = d3['heights']['h_ph']
    print(zz.shape, 'zz')
    if mask_lon is not None:
        if mask_lon[0] is not None and mask_lon[1] is None:
            subset1 = subset1 & (d3['heights']['x_atc'] >= mask_lon[0])
        elif mask_lon[0] is None and mask_lon[1] is not None:
            subset1 = subset1 & (d3['heights']['x_atc'] <= mask_lon[1])
        else:
            subset1 = subset1 & (d3['heights']['x_atc'] >= min(mask_lon)) & (d3['heights']['x_atc'] <= max(mask_lon))
    x_act = d3['heights']['x_atc'][subset1]
    h = d3['heights']['h_ph'][subset1]
    
    signal_conf_ph = d3['heights']['signal_conf_ph'][subset1]
    lat = d3['heights']['lat_ph'][subset1]
    lon = d3['heights']['lon_ph'][subset1]
    ref_elev = d3['geolocation']['ref_elev_all'][subset1]
    print(d3['heights'])
    print(d3.keys())
    print(d3['heights'].keys(), 'height')
    print(d3['geolocation'].keys(), 'location key')
    print(d3['bckgrd_atlas'].keys(), 'bckgrd_atlas')
    print(d3['heights']['quality_ph'], 'quality_ph')
    del d3, subset1
    return x_act, h, signal_conf_ph, lat, lon, ref_elev

def read_all_beam_coordinate(filepath, mask_lat, mask_lon):
    """
    读取所有波束的数据
    :param filepath:
    :param mask_lat:
    :param mask_lon:
    :return:
    """
    atl03_file = glob(filepath)
    is2_atl03_mds = read_hdf5_atl03_coordinate(atl03_file[0])

    # 禁止加载全部数据
    # if mask_lat is None or len(mask_lat) == 0 or mask_lon is None or len(mask_lon) == 0:
    #     return False

    d3 = is2_atl03_mds
    if mask_lon is None and mask_lat is None:
        # 加载全部数据
        return d3
    for beam in is2_atl03_mds.keys():
        subset1 = (d3[beam]['lat'] >= min(mask_lat)) & (d3[beam]['lat'] <= max(mask_lat))
        subset1 = subset1 & (d3[beam]['lon'] >= min(mask_lon)) & (d3[beam]['lon'] <= max(mask_lon))
        d3[beam]['lat'] = d3[beam]['lat'][subset1]
        d3[beam]['lon'] = d3[beam]['lon'][subset1]
    return d3

def read_hdf5_atl08(filename, beam, verbose=False):
    file_id = h5py.File(os.path.expanduser(filename), 'r')

    # 输出HDF5文件信息
    if verbose:
        print(file_id.filename)
        print(list(file_id.keys()))
        print(list(file_id['METADATA'].keys()))
    # 为ICESat-2 ATL08变量和属性分配python字典
    atl08_mds = {}

    # 读取文件中每个输入光束
    beams = [k for k in file_id.keys() if bool(re.match('gt\\d[lr]', k))]
    if beam not in beams:
        print('请填入正确的光束代码')
        return
    # atl08_land_segement = {}
    # for key, val in file_id[beam]['land_segments'].items():
    #     atl08_land_segement['land_segments'][key] = val[:]
    atl08_mds['land_segments'] = {}
    atl08_mds['signal_photons'] = {}
    print(atl08_mds)
    # -- ICESat-2 Geolocation Group
    for key, val in file_id[beam]['signal_photons'].items():
        print(val) 
        atl08_mds['signal_photons'][key] = val[:]

    return atl08_mds

def select_atl03_data(atl03_data, mask, TF_height_range):
    """
    选择数据范围
    Args:
        atl03_data: 所有数据
        mask (list): 维度范围
    Returns:
    """
    # 选择范围
    d3 = atl03_data
    subset1 = (d3['heights']['lon_ph'] > min(mask[0])) & (d3['heights']['lon_ph'] < max(mask[0])) & (d3['heights']['lat_ph'] > min(mask[1])) & (d3['heights']['lat_ph'] < max(mask[1])) & (d3['classed_pc_flag'] == 1) & (d3['heights']['h_ph'] > TF_height_range[0]) & (d3['heights']['h_ph'] < TF_height_range[1]) 

    x_act = d3['heights']['x_atc'][subset1]
    h = d3['heights']['h_ph'][subset1]
    signal_conf_ph = d3['heights']['signal_conf_ph'][subset1]
    lat = d3['heights']['lat_ph'][subset1]
    lon = d3['heights']['lon_ph'][subset1]
    classed_pc_flag = d3['classed_pc_flag'][subset1]

    return x_act, h, signal_conf_ph, lat, lon, classed_pc_flag

def get_atl03_data(filepath, beam):
    """
    读取ATL03数据，根据维度截取数据
    Args:
        filepath (str): h5文件路径
        beam (str): 光束
    Returns:
        返回沿轨道距离，高程距离，光子置信度
    """
    atl03_file = glob.glob(filepath)
    is2_atl03_mds = read_hdf5_atl03_beam_h5py(atl03_file[0], beam=beam, verbose=False)
    # 添加沿轨道距离到数据中
    get_atl03_x_atc(is2_atl03_mds)
    return is2_atl03_mds

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import re

def get_atl_group(folder_path_03, folder_path_08):

    alt03_file_list = glob.glob(os.path.join(folder_path_03, '*ATL03*.h5'))
    n = len(alt03_file_list)

    ATL_name_group = []
    for i in range(n):
        
        # find keyword (date and orbit number)
        file_path = alt03_file_list[i]
        time_match = re.search(r'ATL03_(\d{14})_', file_path)
        orbit_match = re.search(r'ATL03_\d{14}_(\d{8})_', file_path)
        Imaging_date = time_match.group(1)[:8]
        orbit_num = orbit_match.group(1)

        # search target ATL03/08 file with the same keyword (data)
        alt03_file_list_ = glob.glob(os.path.join(folder_path_03, f'*ATL03*{Imaging_date}*{orbit_num}*.h5'))
        alt08_file_list = glob.glob(os.path.join(folder_path_08, f'*ATL08*{Imaging_date}*{orbit_num}*.h5'))
        
        # Construct ATL_pair and storage in ATL_name_group( 2-D List)
        if (len(alt08_file_list) == 1) & (len(alt08_file_list) == 1):
            ATL_pair = [alt03_file_list_[0], alt08_file_list[0]]
            ATL_name_group.append(ATL_pair)

    return ATL_name_group

def find_anomaly(indices_list):
    b = []
    if len(indices_list) <= 5:
        value = 100
    else:
        for i in range(1,len(indices_list)):
            b.append(indices_list[i] - indices_list[i-1])
            
        if max(b) > 5:
            index_max = [index for index, value in enumerate(b) if value == max(b)]
            value = indices_list[index_max[0]+1]
        else:
            value = indices_list[0]
        
    return value

def sea_height_cal(input, height_name, derivative_thred):
    
    if  isinstance(input, str):
        data = pd.read_csv(input)
        df = pd.DataFrame(data)
    else:
        df = input
    
    percentile, derivative, derivative_2th = [], [0], [0]
    height_TS = pd.DataFrame()
    
    if len(df) < 100:
        return 0, height_TS
    else:
        
        for i in range(0, 101):
            p = np.percentile(df[height_name], i)
            percentile.append(p)

        height_TS['percentile'] = percentile
        
        height_TS['percentile'][:5] = height_TS['percentile'][5]
        
        short_window, long_window = 5, 10
        height_TS['EMA5'] = height_TS['percentile'].ewm(span=short_window, adjust=False).mean()
        height_TS['EMA10'] = height_TS['percentile'].ewm(span=long_window, adjust=False).mean()
        height_TS['MACD'] = height_TS['EMA5'] - height_TS['EMA10']

        indices = [index for index, value in enumerate(height_TS['MACD']) if value > derivative_thred]
        target_index = find_anomaly(indices)
        # if indices == []:
        #     indices = [100]

        print(indices[:10], 'indices')
        print(target_index, 'target_index')
        # sea_height = percentile[target_index]
        sea_height = percentile[target_index]
        
        return  sea_height, df#, height_TS, target_index
    
def sea_height_cal_for_view(input, height_name, derivative_thred):
    
    if  isinstance(input, str):
        data = pd.read_csv(input)
        df = pd.DataFrame(data)
    else:
        df = input
    
    percentile, derivative, derivative_2th = [], [0], [0]
    height_TS = pd.DataFrame()
    
    if len(df) < 100:
        return 0, height_TS
    else:
        
        for i in range(0, 101):
            p = np.percentile(df[height_name], i)
            percentile.append(p)

        height_TS['percentile'] = percentile
        short_window, long_window = 5, 10
        # height_TS['EMA5'] = height_TS['percentile'].ewm(span=short_window, adjust=False).mean()
        # height_TS['EMA10'] = height_TS['percentile'].ewm(span=long_window, adjust=False).mean()
        # height_TS['MACD'] = height_TS['EMA5'] - height_TS['EMA10']
        
        height_TS['percentile'][:5] = height_TS['percentile'][5]

    #----------------------------------------------------------------
        height_TS['EMA5'] = height_TS['percentile'].ewm(span=short_window, adjust=False).mean()
        height_TS['EMA10'] = height_TS['percentile'].ewm(span=long_window, adjust=False).mean()
        height_TS['MACD'] = height_TS['EMA5'] - height_TS['EMA10']
        # height_TS['MACD'] = height_TS['Z'].ewm(span=5, adjust=False).mean()
        # height_TS['DEA'] =  height_TS['DIF'].ewm(span=25, adjust=False).mean()
        # height_TS['MACD'] = height_TS['DIF'] - height_TS['DEA']
    
    #----------------------------------------------------------------
    
        indices = [index for index, value in enumerate(height_TS['MACD']) if value > derivative_thred]
        target_index = find_anomaly(indices)
        # if indices == []:
        #     indices = [100]

        print(indices[:10], 'indices')
        print(target_index, 'target_index')
        #sea_height = percentile[target_index]
        sea_height = percentile[target_index]
        return  sea_height, df , height_TS, target_index

In [18]:
import rasterio
def icesat_gt_processing3_new(No ,alt03_path, alt08_path, beam, region_mask, TF_height_range, gap_distance, sea_thred, is_display = False):
    
    
    osgm_path = 'E:\EGM2008_ChinaCoast.tif'
    df_sea_point_remove_list = []
    local_strip_height_list = []
    log = []
    
    for i in range(len(beam)):
        
        df_list = []
        sea_height_list = []
        
        for j in range(len(beam[i])):
            
            atl03_data = get_atl03_data(alt03_path, beam[i][j])
            add_atl08_classed_flag(alt08_path, beam[i][j], atl03_data)
            x_origin, y_origin, conf, lat, lon, classed_pc_flag = select_atl03_data(atl03_data, region_mask, TF_height_range)
            z = pd.DataFrame()
            z['X_Along_track'], z['height'], z['latitude'], z['longtitude'] = x_origin, y_origin, lat, lon
            z['distance'] = (z['X_Along_track'] // gap_distance) * gap_distance
            re_accumulate =  z.groupby('distance').agg(
                
                mean_height = ('height', 'mean'),
                mean_lon = ('longtitude', 'mean'),
                mean_lat = ('latitude', 'mean')
            ).reset_index()


            if osgm_path is not None and not re_accumulate.empty:
                
                
                raster = rasterio.open(osgm_path)
                transformer = Transformer.from_crs(4326, raster.crs, always_xy=True)
                xs, ys = transformer.transform(re_accumulate['mean_lon'].values, re_accumulate['mean_lat'].values)
                coords = list(zip(xs, ys))
                re_accumulate['OSGM'] = [val[0] if val is not None else None for val in raster.sample(coords)]
                re_accumulate['OH'] = re_accumulate['mean_height'] - re_accumulate['OSGM']
                
                # remove land 
                mask = (re_accumulate['OH'] < 10) & (re_accumulate['OH'] > -10) 
                re_accumulate = re_accumulate[mask]
            
            if re_accumulate.empty | len(re_accumulate) < 200:
                print('photon less than 200!')
                continue

            sea_height, df = sea_height_cal(re_accumulate, 'mean_height', sea_thred)  
            df['Strip_No'] =  str(No) + '_' + beam[i][j]
        
            if sea_height > 0:
                sea_height_list.append(sea_height)
                df_list.append(df)  
             
        if sea_height_list == []:
            # empty = pd.DataFrame()
            # return empty, [], [], []
            continue
        
        
        #--------------------------------------------------------------------------------------------------------------  
        
        log.append('sea height list:' + str(sea_height_list) + ' --- ')

        # calculate Final_sea_height
        
        Final_sea_height = np.max(sea_height_list)
        local_strip_height_list.append(Final_sea_height)
        
        log.append('Final_sea_height :'+ str(Final_sea_height) + '\n')
        
        
        #---------------------------------------------------------------------------------------------------------------
    
        # Apply Final_sea_height for all beams
        for k in range(len(df_list)):
            mask = df_list[k]['mean_height'] > Final_sea_height
            print(type(mask), 'type mask', len(mask), 'mask len')
            
            df_sea_point_remove = df_list[k][mask]
            
            if df_sea_point_remove.empty:
                print(filepath_03 + '_' + beam[i][0] + beam[i][1] + ' is empty')
            
            df_sea_point_remove_list.append(df_sea_point_remove)
            
            df_list[k]['sea_height'] = Final_sea_height
            df_list[k]['sea_point'] =  df_list[k]['mean_height'] <= Final_sea_height
            
            if is_display:
            
                plt.figure(figsize=(10, 3))
                plt.plot(df_list[k]['mean_height'], color ='gray',label = 'height')
                plt.plot(df_list[k]['sea_height'], color = 'blue' , linestyle = '--', label = 'sea_height')
                plt.scatter(df_list[k].index[df_list[k]['sea_point']], df_list[k]['mean_height'][df_list[k]['sea_point']], color = 'red', label = 'sea points', zorder = 0.5)
                plt.legend()
                plt.ylabel('height')
                plt.xlabel('photons')
                plt.show()
                #############
        
        merged_df = pd.concat(df_sea_point_remove_list, ignore_index=True)
        
    if len(local_strip_height_list) == 0:
        empty = pd.DataFrame()
        return empty, [], [], []
        
    return merged_df, sea_height_list, Final_sea_height, log

In [ ]:
if __name__ == '__main__':
    

    #----------------------------------------------------------------------------------------
    #--------------------------------------Region 1 -----------------------------------------
    year_list = [2019, 2020, 2021, 2022, 2023, 2024]
    Region_info = {
            'region1': {'name':'xx' , 'mask':[[120.8878,122.02489], [31.6669, 32.5879]]},
            'region2': {'name':'xx', 'mask':[[120.6452, 121.5240], [32.5787, 33.4057]]},
            'region3': {'name':'xx', 'mask':[[120.2037, 120.7612], [33.4031, 34.2791]]},
            'region4': {'name':'xx', 'mask':[[119.4338, 120.3182], [34.2734, 34.7013]]},
            'region5': {'name':'xx', 'mask':[[119.0579, 119.5523], [34.6975, 35.1367]]},
        }
    detect_region_list = ['region4', 'region5']

    for detect_region in detect_region_list:
        print('region:' ,Region_info[detect_region]['name'])
        print('region_mask:', Region_info[detect_region]['mask'])

        region_name = Region_info[detect_region]['name']
        mask = Region_info[detect_region]['mask']
        TF_height_range = [-30, 40]
        #----------------------------------------------------------------------------------------
        for j in range(len(year_list)):

            year = year_list[j]

            output_root_path = f'H:\China_ICESAT-2\JS_SH_ICESAT\JS_SH_{region_name}'
            if not os.path.exists(output_root_path):
                os.makedirs(output_root_path)
                output_path = f'H:\China_ICESAT-2\JS_SH_ICESAT\JS_SH_{region_name}\A_{region_name}_30m_{year}_new10.csv'
            else:
                output_path = f'H:\China_ICESAT-2\JS_SH_ICESAT\JS_SH_{region_name}\A_{region_name}_30m_{year}_new10.csv'

            log_path =  f'H:\China_ICESAT-2\JS_SH_ICESAT\JS_SH_{region_name}\A_log_{region_name}_{year}_new10.txt'
            

            folder_path_03  = f'H:\China_ICESAT-2\JS_SH_ICESAT\JS_SH_{year}\ATL03_JS_SH_{year}'
            folder_path_08 =  f'H:\China_ICESAT-2\JS_SH_ICESAT\JS_SH_{year}\ATL08_JS_SH_{year}' 
            #beam = ['gt1l', 'gt2l', 'gt3l', 'gt1r', 'gt2r', 'gt3r']
            beam = [['gt1l', 'gt1r'], ['gt2l','gt2r'], ['gt3l','gt3r']]

            ATL_name_group = get_atl_group(folder_path_03, folder_path_08)
            
            output = []
            log_list = []
            for i in range(len(ATL_name_group)):
            # for i in range(3):
            # for i in range(1,2):
                
                filepath_03 = ATL_name_group[i][0]
                filepath_08 = ATL_name_group[i][1]
                
                merged_df, _, _, log = icesat_gt_processing3_new(i, filepath_03, filepath_08, beam, mask, TF_height_range , 10, sea_thred = 0.05, is_display = False)
                
                if merged_df.empty:
                    continue
                
                print('------------------------------------')
                output.append(merged_df)
                print('No_'+str(i) + '_' + str(filepath_03) + ' is done')
                print(str(round(i*100/len(ATL_name_group), 2))+ '%' + ' is done')
                log.append('No_'+str(i) + ': ' + str(filepath_03) + '\n')
                log_list.append(log)
            #-----------------------------output log info-------------------------------------- 
            
            with open(log_path, "w") as file:
                for logs in log_list:
                    for log in logs:
                        file.write(log)
            #-------------------------- output result-------------------------------------------
                
            output = pd.concat(output, ignore_index=True) 
            print('output_len: ',len(output))
            print(output[:10])
            output.to_csv(output_path)